<a href="https://colab.research.google.com/github/SANGHATI23/genomic-evidence-reliability/blob/main/22_GES_Aware_Genomic_RAG_Cell_7C15_Aggregation_Performance_and_Bootstrap_Authorization_V3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Not running in Google Colab; Drive mount skipped.')

ROOT = Path('/content/drive/MyDrive/GES_RAG_Temporal_Study')
if not ROOT.exists():
    raise FileNotFoundError(
        f'Project root not found: {ROOT}\n'
        'Confirm Google Drive is mounted and the project directory is unchanged.'
    )

print(f'Project root: {ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/GES_RAG_Temporal_Study


## 1. Imports, exact Cell 7C14 lineage, and fail-closed output paths

In [2]:
from __future__ import annotations

from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
import hashlib
import json
import re
import tempfile

import pandas as pd
import pyarrow.parquet as pq


NOTEBOOK_NAME = (
    '22_GES_Aware_Genomic_RAG_Cell_7C15_'
    'Aggregation_Performance_and_Bootstrap_Authorization.ipynb'
)
CELL_ID = '7C15'
STAGE = '7C'
AMENDMENT_ID = 'A004'
PACKAGE_VERSION = 'v1'
CREATED_UTC = datetime.now(timezone.utc).isoformat()

EXPECTED_RESPONSES = 1_440
EXPECTED_QUESTIONS = 80
EXPECTED_CONDITIONS = 6
EXPECTED_RUNS = 3
EXPECTED_QUESTION_CONDITION_UNITS = 480
BOOTSTRAP_REPLICATES = 2_000

EXPECTED_CELL_7C14_TERMINAL_DECISION = (
    'PASS_STAGE7C14_CONDITION_RESTORATION_COMPLETE_1440_FROZEN_CELL7C12_RESPONSE_'
    'OUTCOMES_JOINED_ONE_TO_ONE_TO_VERIFIED_CELL7C8_ROUTING_AND_FROZEN_7B4_ALIAS_'
    'MAPPING_SIX_CONDITIONS_A_TO_F_AND_RUNS_0_1_2_RESTORED_80X6X3_STRUCTURE_'
    'VERIFIED_UNBLINDED_RESPONSE_LEVEL_TABLE_FROZEN_CHECKSUM_PROTECTED_NO_RUN_'
    'AGGREGATION_CONDITION_LEVEL_PERFORMANCE_ARM_COMPARISON_BOOTSTRAP_OR_'
    'INFERENCE_NEXT_AUTOMATED_EXECUTION_NOT_AUTHORIZED'
)

# --------------------------------------------------------------------------------------
# Exact successful Cell 7C14 package.
# --------------------------------------------------------------------------------------
CELL_7C14_DATA_DIR = (
    ROOT / 'data_processed' / 'stage7_rag'
    / 'cell_7c14_a004_unblinded_response_level_table_v1'
)
CELL_7C14_CONFIG_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c14_a004_unblinded_response_level_table_v1'
)
CELL_7C14_QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c14_a004_unblinded_response_level_table_v1'
)

CELL_7C14 = OrderedDict([
    ('unblinded_response_level_table', {
        'path': CELL_7C14_DATA_DIR / 'cell_7c14_a004_unblinded_response_level_outcomes_v1.parquet',
        'sha256': '101bd5b16730a18b490894ed9a91d5caaface93e4c5c3127b8188e3022d9eee1',
    }),
    ('condition_mapping_snapshot', {
        'path': CELL_7C14_CONFIG_DIR / 'cell_7c14_condition_mapping_snapshot_v1.csv',
        'sha256': '4c7496fae82b50a44940a1de98b22f6345d21cca1ac5715ebf263cb762d21ab1',
    }),
    ('input_inventory', {
        'path': CELL_7C14_CONFIG_DIR / 'cell_7c14_verified_input_inventory_v1.csv',
        'sha256': 'c39cef1ac8d79c80fde8895fb27225427f0c1e2eb2dafe6c698426a7d8a430bf',
    }),
    ('execution_report', {
        'path': CELL_7C14_QC_DIR / 'cell_7c14_condition_restoration_execution_report_v1.json',
        'sha256': '24bb48a3d06b1b2f5871eb804b3055502e2710dc00063d54475d6c5656b31f1e',
    }),
    ('qc', {
        'path': CELL_7C14_QC_DIR / 'cell_7c14_condition_restoration_qc_v1.json',
        'sha256': 'e10ff4f24a1ce58b42f97b2e8a4a0adb595837e407aaa293bc00b9df2a6bf12c',
    }),
    ('manifest', {
        'path': CELL_7C14_CONFIG_DIR / 'cell_7c14_unblinded_response_table_manifest_v1.json',
        'sha256': 'a0691be85e714d259cd863c86a5bb1f8fbfbe8d9a65a221a13d99bd30dfb84fd',
    }),
])

# --------------------------------------------------------------------------------------
# Frozen A004 endpoint + aggregation/inference specifications.
# --------------------------------------------------------------------------------------
A004_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'protocol_amendment_A004_fully_automated_structured_evaluation_v1'
)

A004_ENDPOINT_SPEC = {
    'path': A004_DIR / 'protocol_amendment_A004_automated_endpoint_spec_v1.json',
    'sha256': '40cf8c54be9aa0337240caa6f6c1f8a31d0baf3078c94638fe8bf79e212edb97',
}
A004_AGGREGATION_SPEC = {
    'path': A004_DIR / 'protocol_amendment_A004_aggregation_and_inference_spec_v1.json',
    'sha256': 'b4ea6ae437edfbdc626d249855203c2b6f9b2b6131212d48424760cdeaf6d1f9',
}

BOOTSTRAP_SEED = int(A004_AGGREGATION_SPEC['sha256'][:8], 16)

# --------------------------------------------------------------------------------------
# Exact frozen primary-question artifact for gene metadata.
# --------------------------------------------------------------------------------------
PRIMARY_QUESTION_FILENAME = 'cell_7b3_primary_question_set_v1.csv'
PRIMARY_QUESTION_SHA256 = (
    'c76e81952fcc6a698866b64da7b7daabeb281b7b9d10e17873596095b69d95df'
)

QUESTION_SEARCH_ROOTS = [
    ROOT,
]

# --------------------------------------------------------------------------------------
# Cell 7C15 output package.
# --------------------------------------------------------------------------------------
OUT_DIR = (
    ROOT / 'configs' / 'stage7_rag'
    / 'cell_7c15_aggregation_performance_bootstrap_authorization_v1'
)
QC_DIR = (
    ROOT / 'outputs' / 'quality_checks' / 'stage7_rag'
    / 'cell_7c15_aggregation_performance_bootstrap_authorization_v1'
)

OUTPUTS = OrderedDict([
    ('authorization',
     OUT_DIR / 'cell_7c15_aggregation_performance_bootstrap_authorization_v1.json'),
    ('analysis_plan_snapshot',
     OUT_DIR / 'cell_7c15_authorized_analysis_plan_snapshot_v1.json'),
    ('verified_input_inventory',
     OUT_DIR / 'cell_7c15_verified_input_inventory_v1.csv'),
    ('qc',
     QC_DIR / 'cell_7c15_aggregation_performance_bootstrap_authorization_qc_v1.json'),
    ('manifest',
     OUT_DIR / 'cell_7c15_aggregation_performance_bootstrap_authorization_manifest_v1.json'),
])

for directory in (OUT_DIR, QC_DIR):
    directory.mkdir(parents=True, exist_ok=True)

existing = [str(path) for path in OUTPUTS.values() if path.exists()]
if existing:
    raise FileExistsError(
        'Cell 7C15 fail-closed overwrite protection is active. Existing output(s):\\n- '
        + '\\n- '.join(existing)
    )

print(f'Authorization directory: {OUT_DIR}')
print(f'QC directory           : {QC_DIR}')
print(f'Frozen bootstrap seed   : {BOOTSTRAP_SEED}')

Authorization directory: /content/drive/MyDrive/GES_RAG_Temporal_Study/configs/stage7_rag/cell_7c15_aggregation_performance_bootstrap_authorization_v1
QC directory           : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/quality_checks/stage7_rag/cell_7c15_aggregation_performance_bootstrap_authorization_v1
Frozen bootstrap seed   : 3035261668


## 2. SHA-256, sidecar, stable-write, and artifact-location helpers

In [3]:
def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def sidecar_path(path: Path) -> Path:
    return path.with_name(path.name + '.sha256')


def read_sidecar_hash(path: Path) -> str:
    text = path.read_text(encoding='utf-8').strip()
    if not text:
        raise ValueError(f'Empty SHA-256 sidecar: {path}')
    token = text.split()[0].strip()
    if not re.fullmatch(r'[0-9a-fA-F]{64}', token):
        raise ValueError(f'Invalid SHA-256 sidecar: {path}')
    return token.lower()


def sidecar_is_valid(path: Path) -> bool:
    return (
        path.exists()
        and sidecar_path(path).exists()
        and read_sidecar_hash(sidecar_path(path)) == sha256_file(path)
    )


def verify_exact_artifact(
    label: str,
    path: Path,
    expected_sha256: str,
    require_sidecar: bool = True,
) -> dict[str, Any]:
    if not path.exists():
        raise FileNotFoundError(f'Missing frozen artifact [{label}]: {path}')
    observed = sha256_file(path)
    if observed != expected_sha256:
        raise AssertionError(
            f'{label} SHA-256 mismatch.\\nExpected: {expected_sha256}\\nObserved: {observed}'
        )
    if require_sidecar and not sidecar_is_valid(path):
        raise AssertionError(f'Invalid/missing sidecar for {label}: {path}')
    return {
        'input_id': label,
        'path': str(path),
        'sha256': observed,
        'bytes': int(path.stat().st_size),
        'sidecar_path': str(sidecar_path(path)) if sidecar_path(path).exists() else '',
        'sidecar_valid': sidecar_is_valid(path) if sidecar_path(path).exists() else False,
    }


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding='utf-8'))


def stable_write_json(path: Path, payload: Any) -> str:
    path.write_text(
        json.dumps(
            payload,
            indent=2,
            sort_keys=True,
            ensure_ascii=False,
            allow_nan=False,
        ) + chr(10),
        encoding='utf-8',
    )
    return sha256_file(path)


def stable_write_csv(path: Path, frame: pd.DataFrame) -> str:
    frame.to_csv(path, index=False, encoding='utf-8', lineterminator=chr(10))
    return sha256_file(path)


def write_sidecar(path: Path) -> None:
    sidecar_path(path).write_text(
        f'{sha256_file(path)}  {path.name}' + chr(10),
        encoding='utf-8',
    )


def locate_unique_named_artifact(
    filename: str,
    expected_sha256: str,
    roots: list[Path],
) -> Path:
    """
    Locate a frozen artifact robustly without assuming its historical directory.

    Search order:
      1) exact filename anywhere under the supplied roots;
      2) .sha256 sidecars whose first token equals the expected hash;
      3) candidate CSV files with "question" in the name under the supplied roots,
         verified by exact SHA-256.

    The function returns only when exactly one exact-hash artifact is found.
    """
    exact_matches: list[Path] = []
    seen: set[Path] = set()

    def consider(path: Path) -> None:
        resolved = path.resolve()
        if resolved in seen or not path.is_file():
            return
        seen.add(resolved)
        try:
            if sha256_file(path) == expected_sha256:
                exact_matches.append(path)
        except OSError:
            return

    # 1) Exact historical filename anywhere under the project.
    for root in roots:
        if not root.exists():
            continue
        for path in root.rglob(filename):
            consider(path)

    # 2) Locate from SHA sidecars even when the file lives in an unexpected directory.
    for root in roots:
        if not root.exists():
            continue
        for sidecar in root.rglob('*.sha256'):
            try:
                token = sidecar.read_text(encoding='utf-8').strip().split()[0].lower()
            except Exception:
                continue
            if token != expected_sha256.lower():
                continue

            # Standard sidecar convention used throughout the project:
            # <artifact_filename>.sha256
            if sidecar.name.endswith('.sha256'):
                artifact_name = sidecar.name[:-7]
                candidate = sidecar.with_name(artifact_name)
                if candidate.exists():
                    consider(candidate)

    # 3) Fallback: hash only plausible question CSV artifacts, not the entire repository.
    if not exact_matches:
        for root in roots:
            if not root.exists():
                continue
            for path in root.rglob('*.csv'):
                name = path.name.lower()
                if 'question' in name or '7b3' in name:
                    consider(path)

    unique = []
    unique_resolved = set()
    for path in exact_matches:
        resolved = path.resolve()
        if resolved not in unique_resolved:
            unique.append(path)
            unique_resolved.add(resolved)

    if len(unique) != 1:
        raise RuntimeError(
            f'Expected exactly one artifact with SHA-256 {expected_sha256}; '
            f'found {len(unique)} exact match(es): {[str(x) for x in unique]}'
        )

    return unique[0]


with tempfile.TemporaryDirectory(prefix='cell_7c15_writer_test_') as tmp:
    p = Path(tmp) / 'x.json'
    stable_write_json(p, {'ok': True})
    write_sidecar(p)
    assert load_json(p) == {'ok': True}
    assert sidecar_is_valid(p)

print('Serialization / SHA-256 helper self-test: PASS')

Serialization / SHA-256 helper self-test: PASS


## 3. Reverify the complete Cell 7C14 unblinded response-level package

In [4]:
verified_inputs = []

for artifact_id, spec in CELL_7C14.items():
    record = verify_exact_artifact(
        f'cell_7c14_{artifact_id}',
        spec['path'],
        spec['sha256'],
        require_sidecar=True,
    )
    record['source_cell'] = '7C14'
    verified_inputs.append(record)

manifest_7c14 = load_json(CELL_7C14['manifest']['path'])
qc_7c14 = load_json(CELL_7C14['qc']['path'])
report_7c14 = load_json(CELL_7C14['execution_report']['path'])

if manifest_7c14.get('terminal_decision') != EXPECTED_CELL_7C14_TERMINAL_DECISION:
    raise AssertionError('Cell 7C14 terminal PASS mismatch.')
if manifest_7c14.get('next_authorized_cell') is not None:
    raise AssertionError('Cell 7C14 unexpectedly authorized a downstream cell.')
if manifest_7c14.get('run_aggregation_authorized') is not False:
    raise AssertionError('Cell 7C14 unexpectedly authorized run aggregation.')
if manifest_7c14.get('condition_level_metric_calculation_authorized') is not False:
    raise AssertionError('Cell 7C14 unexpectedly authorized condition-level metrics.')
if manifest_7c14.get('arm_comparison_authorized') is not False:
    raise AssertionError('Cell 7C14 unexpectedly authorized arm comparison.')
if manifest_7c14.get('bootstrap_inference_authorized') is not False:
    raise AssertionError('Cell 7C14 unexpectedly authorized bootstrap inference.')
if int(qc_7c14.get('failed_checks', -1)) != 0:
    raise AssertionError('Cell 7C14 QC does not report zero failures.')

# Inspect identity columns only; do not read scientific metric values.
identity_columns = [
    'review_item_id',
    'question_id',
    'blinded_alias',
    'run_id',
    'condition_id',
    'condition_name',
]
identity = pd.read_parquet(
    CELL_7C14['unblinded_response_level_table']['path'],
    columns=identity_columns,
)

if len(identity) != EXPECTED_RESPONSES:
    raise AssertionError('Cell 7C14 unblinded table must contain 1,440 rows.')
if identity['review_item_id'].nunique() != EXPECTED_RESPONSES:
    raise AssertionError('Cell 7C14 review_item_id must remain unique.')
if identity['question_id'].nunique() != EXPECTED_QUESTIONS:
    raise AssertionError('Expected 80 question IDs.')
if set(identity['condition_id'].astype(str)) != {'A', 'B', 'C', 'D', 'E', 'F'}:
    raise AssertionError('Condition IDs are not exactly A-F.')
if set(pd.to_numeric(identity['run_id'], errors='raise').astype(int)) != {0, 1, 2}:
    raise AssertionError('Run IDs are not exactly 0,1,2.')

qcond_counts = identity.groupby(['question_id', 'condition_id']).size()
if len(qcond_counts) != EXPECTED_QUESTION_CONDITION_UNITS:
    raise AssertionError('Expected exactly 480 question-condition units.')
if not qcond_counts.eq(EXPECTED_RUNS).all():
    raise AssertionError('Every question-condition unit must contain exactly 3 runs.')

print('Cell 7C14 package                     : 6/6 exact hashes + sidecars')
print('Cell 7C14 terminal PASS               : VERIFIED')
print('Unblinded response rows               : 1,440')
print('Question-condition units              : 480')
print('Runs per question-condition           : 3')
print('Scientific metric values inspected    : NO')

Cell 7C14 package                     : 6/6 exact hashes + sidecars
Cell 7C14 terminal PASS               : VERIFIED
Unblinded response rows               : 1,440
Question-condition units              : 480
Runs per question-condition           : 3
Scientific metric values inspected    : NO


## 4. Reverify the frozen A004 endpoint and aggregation/inference specifications

In [5]:
for label, spec in [
    ('cell_7c11_A004_endpoint_spec', A004_ENDPOINT_SPEC),
    ('cell_7c11_A004_aggregation_inference_spec', A004_AGGREGATION_SPEC),
]:
    record = verify_exact_artifact(
        label,
        spec['path'],
        spec['sha256'],
        require_sidecar=True,
    )
    record['source_cell'] = '7C11'
    verified_inputs.append(record)

endpoint_spec = load_json(A004_ENDPOINT_SPEC['path'])
aggregation_spec = load_json(A004_AGGREGATION_SPEC['path'])

if endpoint_spec['primary_automated_endpoint']['name'] != 'automated_evidence_fidelity_pass':
    raise AssertionError('Frozen A004 primary endpoint changed.')
if aggregation_spec['primary_comparison']['experimental_condition'] != 'D Full-GES':
    raise AssertionError('Frozen primary experimental condition changed.')
if aggregation_spec['primary_comparison']['reference_condition'] != 'A semantic-only':
    raise AssertionError('Frozen primary reference condition changed.')
if aggregation_spec['bootstrap']['unit'] != 'question':
    raise AssertionError('Frozen bootstrap unit changed.')
if aggregation_spec['bootstrap']['paired'] is not True:
    raise AssertionError('Frozen bootstrap pairing changed.')
if aggregation_spec['bootstrap']['replicates'] != BOOTSTRAP_REPLICATES:
    raise AssertionError('Frozen bootstrap replicate count changed.')
if len(aggregation_spec['mandatory_secondary_comparisons']) != 4:
    raise AssertionError('Frozen mandatory secondary comparison family changed.')

authorized_secondary_metric_inference = set(
    aggregation_spec['secondary_metric_inference'].keys()
)

expected_secondary_metric_inference = {
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
    'context_citation_coverage',
    'invalid_or_hallucinated_evidence_id_count',
    'confidence',
}
if authorized_secondary_metric_inference != expected_secondary_metric_inference:
    raise AssertionError(
        'Frozen secondary metric inference membership changed.\\n'
        f'Observed: {sorted(authorized_secondary_metric_inference)}\\n'
        f'Expected: {sorted(expected_secondary_metric_inference)}'
    )


print('A004 endpoint specification           : VERIFIED')
print('A004 aggregation/inference spec        : VERIFIED')
print('Primary comparison                    : D Full-GES vs A semantic-only')
print('Mandatory secondary arm comparisons   : D vs B/C/E/F')
print('Paired bootstrap                      : 2,000 question-level replicates')
print(f'Deterministic bootstrap seed           : {BOOTSTRAP_SEED}')

A004 endpoint specification           : VERIFIED
A004 aggregation/inference spec        : VERIFIED
Primary comparison                    : D Full-GES vs A semantic-only
Mandatory secondary arm comparisons   : D vs B/C/E/F
Paired bootstrap                      : 2,000 question-level replicates
Deterministic bootstrap seed           : 3035261668


## 5. Reverify the exact frozen 80-question metadata and gene reporting boundary

In [6]:
primary_question_path = locate_unique_named_artifact(
    PRIMARY_QUESTION_FILENAME,
    PRIMARY_QUESTION_SHA256,
    QUESTION_SEARCH_ROOTS,
)

primary_question_record = verify_exact_artifact(
    'cell_7b3_primary_question_set',
    primary_question_path,
    PRIMARY_QUESTION_SHA256,
    require_sidecar=False,
)
primary_question_record['source_cell'] = '7B3'
verified_inputs.append(primary_question_record)

questions = pd.read_csv(primary_question_path, dtype=str).fillna('')

required_question_columns = {'question_id', 'target_gene'}
missing_question_columns = sorted(required_question_columns - set(questions.columns))
if missing_question_columns:
    raise AssertionError(
        'Frozen primary question set missing required columns: '
        + ', '.join(missing_question_columns)
    )

if len(questions) != EXPECTED_QUESTIONS:
    raise AssertionError(f'Expected 80 frozen primary questions; observed {len(questions)}.')
if questions['question_id'].nunique() != EXPECTED_QUESTIONS:
    raise AssertionError('Frozen primary question IDs are not unique.')

question_ids_7c14 = set(identity['question_id'].astype(str))
question_ids_7b3 = set(questions['question_id'].astype(str))
if question_ids_7c14 != question_ids_7b3:
    raise AssertionError('Cell 7C14 question IDs do not exactly match frozen Cell 7B3 primary questions.')

observed_genes = set(questions['target_gene'].astype(str).str.strip())
expected_genes = {'BRCA1', 'BRCA2', 'MLH1', 'EGFR'}
if observed_genes != expected_genes:
    raise AssertionError(
        f'Unexpected frozen target-gene set: {sorted(observed_genes)}'
    )

print(f'Frozen primary question artifact       : {primary_question_path}')
print('Primary question rows                  : 80')
print('Question ID alignment                  : EXACT')
print('Frozen target genes                    : BRCA1, BRCA2, MLH1, EGFR')
print('EGFR reporting                         : exploratory / separate')
print('Gene-level hypothesis tests added      : NO')

Frozen primary question artifact       : /content/drive/MyDrive/GES_RAG_Temporal_Study/outputs/tables/stage7_rag/cell_7b3_primary_question_set_v1.csv
Primary question rows                  : 80
Question ID alignment                  : EXACT
Frozen target genes                    : BRCA1, BRCA2, MLH1, EGFR
EGFR reporting                         : exploratory / separate
Gene-level hypothesis tests added      : NO


## 6. Freeze exact Cell 7C16 analysis authorization

In [7]:
PRIMARY_ENDPOINT = 'automated_evidence_fidelity_pass'

PRIMARY_COMPARISON = {
    'comparison_id': 'PRIMARY_D_MINUS_A',
    'metric': PRIMARY_ENDPOINT,
    'experimental_condition_id': 'D',
    'reference_condition_id': 'A',
    'experimental_condition_name': 'Full-GES',
    'reference_condition_name': 'semantic-only',
    'direction': 'D_minus_A',
}

MANDATORY_SECONDARY_ARM_COMPARISONS = [
    {
        'comparison_id': 'SECONDARY_D_MINUS_B',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'B',
    },
    {
        'comparison_id': 'SECONDARY_D_MINUS_C',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'C',
    },
    {
        'comparison_id': 'SECONDARY_D_MINUS_E',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'E',
    },
    {
        'comparison_id': 'SECONDARY_D_MINUS_F',
        'metric': PRIMARY_ENDPOINT,
        'experimental_condition_id': 'D',
        'reference_condition_id': 'F',
    },
]

SECONDARY_D_VS_A_INFERENTIAL_METRICS = [
    'citation_integrity_pass',
    'citation_presence_pass',
    'conflict_concordance_pass',
    'caution_policy_concordance_pass',
    'context_citation_coverage',
    'invalid_or_hallucinated_evidence_id_count',
]

DESCRIPTIVE_ONLY_FIELDS = [
    'valid_context_evidence_id_count',
    'distinct_valid_context_evidence_id_count',
    'distinct_invalid_or_hallucinated_evidence_id_count',
    'cited_evidence_id_count',
    'distinct_cited_evidence_id_count',
    'duplicate_cited_evidence_id_count',
    'required_caution_compliance',
    'over_abstention',
    'response_policy',
    'evidence_strength',
    'confidence',
]

analysis_plan_snapshot = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'created_utc': CREATED_UTC,
    'analysis_unit': {
        'input_response_rows': 1440,
        'question_condition_units_after_run_aggregation': 480,
        'runs_per_question_condition': 3,
        'aggregation_rule':
            'arithmetic mean across run_id 0,1,2 within each question_id x condition_id',
    },
    'primary_endpoint': PRIMARY_ENDPOINT,
    'primary_comparison': PRIMARY_COMPARISON,
    'mandatory_secondary_arm_comparisons': MANDATORY_SECONDARY_ARM_COMPARISONS,
    'secondary_D_vs_A_inferential_metrics': SECONDARY_D_VS_A_INFERENTIAL_METRICS,
    'descriptive_only_fields': DESCRIPTIVE_ONLY_FIELDS,
    'bootstrap': {
        'replicates': BOOTSTRAP_REPLICATES,
        'seed': BOOTSTRAP_SEED,
        'seed_derivation':
            'int(first_8_hex_chars_of_frozen_A004_aggregation_spec_sha256, 16)',
        'resampling_unit': 'question_id',
        'paired': True,
        'questions_per_replicate': 80,
        'sampling_with_replacement': True,
        'interval': 'two-sided 95% percentile',
        'point_estimate':
            'mean paired question-level difference using all 80 questions',
        'p_values_required': False,
    },
    'gene_reporting': {
        'join_source_sha256': PRIMARY_QUESTION_SHA256,
        'prespecified_descriptive_genes': ['BRCA1', 'BRCA2', 'MLH1'],
        'exploratory_gene': 'EGFR',
        'EGFR_report_separately': True,
        'gene_specific_hypothesis_testing_authorized': False,
    },
    'permitted_outputs_in_cell_7c16': [
        '480-row question-condition aggregated metric table',
        'condition-level descriptive summary table',
        'primary D-vs-A paired comparison estimate and 95% percentile bootstrap CI',
        'mandatory D-vs-B/C/E/F primary-endpoint comparison estimates and 95% percentile bootstrap CIs',
        'D-vs-A secondary-metric paired estimates and 95% percentile bootstrap CIs',
        'bootstrap replicate audit table',
        'prespecified gene-stratified descriptive summary table with EGFR labeled exploratory',
        'response-policy and evidence-strength descriptive distributions',
        'execution report, QC, manifest, and checksums',
    ],
    'not_authorized': [
        'new endpoints not frozen in A004',
        'new arm comparisons outside D-vs-A/B/C/E/F',
        'gene-specific inferential hypothesis tests',
        'post-hoc subgroup discovery',
        'LLM-as-judge evaluation',
        'human-review substitution',
        'free-text factual correctness claims',
        'semantic citation entailment claims',
    ],
}

stable_write_json(
    OUTPUTS['analysis_plan_snapshot'],
    analysis_plan_snapshot,
)
write_sidecar(OUTPUTS['analysis_plan_snapshot'])

authorization_decision = (
    'AUTHORIZE_STAGE7C_CELL7C16_PRESPECIFIED_A004_RAG_PERFORMANCE_ANALYSIS_FROM_'
    'FROZEN_CELL7C14_UNBLINDED_RESPONSE_LEVEL_TABLE_AGGREGATE_THREE_RUNS_WITHIN_'
    'EACH_OF_480_QUESTION_CONDITION_UNITS_PRIMARY_AUTOMATED_EVIDENCE_FIDELITY_'
    'D_MINUS_A_PRIMARY_D_MINUS_B_C_E_F_MANDATORY_SECONDARY_AND_D_MINUS_A_FROZEN_'
    'SECONDARY_METRICS_WITH_2000_PAIRED_QUESTION_LEVEL_BOOTSTRAP_REPLICATES_'
    'DESCRIPTIVE_GENE_REPORTING_BRCA1_BRCA2_MLH1_AND_EGFR_EXPLORATORY_SEPARATE_'
    'NO_NEW_ENDPOINTS_COMPARISONS_SUBGROUP_TESTS_HUMAN_REVIEW_LLM_JUDGE_FREE_TEXT_'
    'FACTUAL_CORRECTNESS_OR_SEMANTIC_CITATION_ENTAILMENT_CLAIMS'
)

authorization_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'version': '1.0.0',
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'authorization_decision': authorization_decision,
    'authorization_basis': {
        'cell_7c14_manifest_sha256': CELL_7C14['manifest']['sha256'],
        'cell_7c14_unblinded_response_level_table_sha256':
            CELL_7C14['unblinded_response_level_table']['sha256'],
        'A004_endpoint_spec_sha256': A004_ENDPOINT_SPEC['sha256'],
        'A004_aggregation_inference_spec_sha256': A004_AGGREGATION_SPEC['sha256'],
        'primary_question_set_sha256': PRIMARY_QUESTION_SHA256,
        'primary_question_set_path': str(primary_question_path),
    },
    'cell_7c15_operations': {
        'scientific_metric_values_read': False,
        'run_aggregation_performed': False,
        'condition_level_performance_calculated': False,
        'arm_comparison_performed': False,
        'bootstrap_performed': False,
    },
    'next_authorized_cell': '7C16',
    'cell_7c16_scope': {
        'run_aggregation_authorized': True,
        'condition_level_metric_calculation_authorized': True,
        'primary_D_vs_A_authorized': True,
        'mandatory_secondary_D_vs_B_C_E_F_authorized': True,
        'secondary_D_vs_A_metric_inference_authorized': True,
        'bootstrap_replicates': BOOTSTRAP_REPLICATES,
        'bootstrap_seed': BOOTSTRAP_SEED,
        'gene_descriptive_reporting_authorized': True,
        'gene_specific_inferential_testing_authorized': False,
        'new_endpoint_discovery_authorized': False,
        'new_arm_comparisons_authorized': False,
    },
}

prewrite_checks = OrderedDict([
    ('cell7c14_6_artifacts_verified',
     len([x for x in verified_inputs if x['source_cell'] == '7C14']) == 6),
    ('cell7c14_terminal_pass_exact',
     manifest_7c14.get('terminal_decision') == EXPECTED_CELL_7C14_TERMINAL_DECISION),
    ('identity_rows_1440', len(identity) == 1440),
    ('question_condition_units_480', len(qcond_counts) == 480),
    ('three_runs_per_question_condition', qcond_counts.eq(3).all()),
    ('A004_endpoint_spec_verified',
     sha256_file(A004_ENDPOINT_SPEC['path']) == A004_ENDPOINT_SPEC['sha256']),
    ('A004_aggregation_spec_verified',
     sha256_file(A004_AGGREGATION_SPEC['path']) == A004_AGGREGATION_SPEC['sha256']),
    ('primary_question_set_verified',
     sha256_file(primary_question_path) == PRIMARY_QUESTION_SHA256),
    ('question_alignment_exact', question_ids_7c14 == question_ids_7b3),
    ('gene_set_exact', observed_genes == {'BRCA1', 'BRCA2', 'MLH1', 'EGFR'}),
    ('primary_D_minus_A_exact',
     PRIMARY_COMPARISON['experimental_condition_id'] == 'D'
     and PRIMARY_COMPARISON['reference_condition_id'] == 'A'),
    ('mandatory_secondary_count_4',
     len(MANDATORY_SECONDARY_ARM_COMPARISONS) == 4),
    ('secondary_metric_inference_count_6',
     len(SECONDARY_D_VS_A_INFERENTIAL_METRICS) == 6),
    ('bootstrap_replicates_2000', BOOTSTRAP_REPLICATES == 2000),
    ('bootstrap_seed_deterministic',
     BOOTSTRAP_SEED == int(A004_AGGREGATION_SPEC['sha256'][:8], 16)),
    ('gene_inference_false',
     analysis_plan_snapshot['gene_reporting']['gene_specific_hypothesis_testing_authorized'] is False),
    ('no_metrics_calculated_in_7c15',
     authorization_payload['cell_7c15_operations']['condition_level_performance_calculated'] is False),
    ('no_arm_comparison_in_7c15',
     authorization_payload['cell_7c15_operations']['arm_comparison_performed'] is False),
    ('no_bootstrap_in_7c15',
     authorization_payload['cell_7c15_operations']['bootstrap_performed'] is False),
    ('cell7c16_aggregation_true',
     authorization_payload['cell_7c16_scope']['run_aggregation_authorized'] is True),
    ('cell7c16_primary_true',
     authorization_payload['cell_7c16_scope']['primary_D_vs_A_authorized'] is True),
    ('cell7c16_bootstrap_2000',
     authorization_payload['cell_7c16_scope']['bootstrap_replicates'] == 2000),
    ('no_new_endpoint_discovery',
     authorization_payload['cell_7c16_scope']['new_endpoint_discovery_authorized'] is False),
    ('no_new_arm_comparisons',
     authorization_payload['cell_7c16_scope']['new_arm_comparisons_authorized'] is False),
])

failed = [name for name, passed in prewrite_checks.items() if not bool(passed)]
if failed:
    raise RuntimeError(
        'Cell 7C15 authorization QC failed:\\n- ' + '\\n- '.join(failed)
    )

stable_write_json(OUTPUTS['authorization'], authorization_payload)
write_sidecar(OUTPUTS['authorization'])

input_inventory = pd.DataFrame(verified_inputs)
stable_write_csv(OUTPUTS['verified_input_inventory'], input_inventory)
write_sidecar(OUTPUTS['verified_input_inventory'])

terminal_decision = (
    'PASS_STAGE7C15_CELL7C14_UNBLINDED_RESPONSE_TABLE_REVERIFIED_1440_ROWS_480_'
    'QUESTION_CONDITION_UNITS_THREE_RUNS_EACH_A004_ENDPOINT_AND_AGGREGATION_SPECS_'
    'REVERIFIED_80_FROZEN_QUESTION_GENE_METADATA_ALIGNED_CELL7C16_RAG_PERFORMANCE_'
    'ANALYSIS_AUTHORIZED_D_MINUS_A_PRIMARY_D_MINUS_B_C_E_F_SECONDARY_D_MINUS_A_'
    'FROZEN_SECONDARY_METRICS_2000_PAIRED_QUESTION_BOOTSTRAP_DETERMINISTIC_SEED_'
    'GENE_DESCRIPTIVE_ONLY_EGFR_EXPLORATORY_NO_NEW_ENDPOINTS_ARM_COMPARISONS_OR_'
    'SUBGROUP_HYPOTHESIS_TESTS'
)

qc_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'checks': {name: bool(value) for name, value in prewrite_checks.items()},
    'passed_checks': len(prewrite_checks),
    'failed_checks': 0,
    'total_checks': len(prewrite_checks),
    'terminal_decision': terminal_decision,
}
stable_write_json(OUTPUTS['qc'], qc_payload)
write_sidecar(OUTPUTS['qc'])

manifest_payload = {
    'cell_id': CELL_ID,
    'stage': STAGE,
    'protocol_amendment_id': AMENDMENT_ID,
    'package_version': PACKAGE_VERSION,
    'created_utc': CREATED_UTC,
    'notebook': NOTEBOOK_NAME,
    'upstream_lineage': {
        'cell_7c14_manifest_sha256': CELL_7C14['manifest']['sha256'],
        'cell_7c14_unblinded_response_level_table_sha256':
            CELL_7C14['unblinded_response_level_table']['sha256'],
        'A004_endpoint_spec_sha256': A004_ENDPOINT_SPEC['sha256'],
        'A004_aggregation_inference_spec_sha256': A004_AGGREGATION_SPEC['sha256'],
        'primary_question_set_sha256': PRIMARY_QUESTION_SHA256,
    },
    'authorization_decision': authorization_decision,
    'terminal_decision': terminal_decision,
    'next_authorized_cell': '7C16',
    'run_aggregation_authorized_in_7c16': True,
    'condition_level_metric_calculation_authorized_in_7c16': True,
    'primary_D_vs_A_authorized_in_7c16': True,
    'mandatory_secondary_D_vs_B_C_E_F_authorized_in_7c16': True,
    'paired_bootstrap_2000_authorized_in_7c16': True,
    'gene_specific_inferential_testing_authorized_in_7c16': False,
    'new_endpoint_discovery_authorized_in_7c16': False,
    'new_arm_comparisons_authorized_in_7c16': False,
    'output_artifacts': {
        key: {
            'path': str(path),
            'sha256': sha256_file(path),
            'sidecar_valid': sidecar_is_valid(path),
        }
        for key, path in OUTPUTS.items()
        if key != 'manifest'
    },
}

stable_write_json(OUTPUTS['manifest'], manifest_payload)
write_sidecar(OUTPUTS['manifest'])

for path in OUTPUTS.values():
    if not path.exists() or not sidecar_is_valid(path):
        raise AssertionError(f'Cell 7C15 final readback failed: {path}')

rb_auth = load_json(OUTPUTS['authorization'])
rb_plan = load_json(OUTPUTS['analysis_plan_snapshot'])
rb_qc = load_json(OUTPUTS['qc'])
rb_manifest = load_json(OUTPUTS['manifest'])

readback_checks = OrderedDict([
    ('next_cell_7c16', rb_manifest.get('next_authorized_cell') == '7C16'),
    ('aggregation_true',
     rb_manifest.get('run_aggregation_authorized_in_7c16') is True),
    ('condition_metrics_true',
     rb_manifest.get('condition_level_metric_calculation_authorized_in_7c16') is True),
    ('primary_true',
     rb_manifest.get('primary_D_vs_A_authorized_in_7c16') is True),
    ('secondary_arm_family_true',
     rb_manifest.get('mandatory_secondary_D_vs_B_C_E_F_authorized_in_7c16') is True),
    ('bootstrap_true',
     rb_manifest.get('paired_bootstrap_2000_authorized_in_7c16') is True),
    ('gene_inference_false',
     rb_manifest.get('gene_specific_inferential_testing_authorized_in_7c16') is False),
    ('new_endpoints_false',
     rb_manifest.get('new_endpoint_discovery_authorized_in_7c16') is False),
    ('new_arms_false',
     rb_manifest.get('new_arm_comparisons_authorized_in_7c16') is False),
    ('plan_bootstrap_seed_exact',
     rb_plan['bootstrap']['seed'] == BOOTSTRAP_SEED),
    ('plan_primary_exact',
     rb_plan['primary_comparison']['comparison_id'] == 'PRIMARY_D_MINUS_A'),
    ('qc_zero_failures', int(rb_qc.get('failed_checks', -1)) == 0),
    ('all_sidecars_valid', all(sidecar_is_valid(path) for path in OUTPUTS.values())),
])

failed_rb = [name for name, passed in readback_checks.items() if not bool(passed)]
if failed_rb:
    raise RuntimeError(
        'Cell 7C15 final readback QC failed:\\n- ' + '\\n- '.join(failed_rb)
    )

total_checks = len(prewrite_checks) + len(readback_checks)

separator = '=' * 158
print('\\n' + separator)
print('EXPERIMENT 2 — STAGE 7C — CELL 7C15')
print('AGGREGATION, PERFORMANCE, AND PAIRED-BOOTSTRAP AUTHORIZATION')
print(separator)
print(f'Notebook                                      : {NOTEBOOK_NAME}')
print(f'Project root                                  : {ROOT}')

print('\\nCELL 7C14 REVERIFICATION')
print(f'Cell 7C14 manifest SHA-256                    : {CELL_7C14["manifest"]["sha256"]}')
print('Cell 7C14 terminal PASS verified              : YES')
print('Unblinded response rows                       : 1,440')
print('Question-condition units                      : 480')
print('Runs per question-condition                   : 3')
print('Scientific metric values inspected in 7C15   : NO')

print('\\nFROZEN ANALYSIS DESIGN')
print('Primary endpoint                              : automated_evidence_fidelity_pass')
print('Primary comparison                            : D Full-GES minus A semantic-only')
print('Mandatory secondary arm comparisons           : D minus B / C / E / F')
print('Secondary metric inference                    : D minus A only, 6 frozen metrics')
print('Run aggregation                               : mean of runs 0,1,2 per question-condition')
print('Bootstrap                                     : 2,000 paired question-level replicates')
print(f'Bootstrap seed                                : {BOOTSTRAP_SEED}')
print('P-values                                      : NOT REQUIRED')
print('Gene-specific hypothesis tests                : NO')
print('EGFR                                          : exploratory / separate')

print('\\nCELL 7C16 AUTHORIZATION')
print('Three-run aggregation                         : AUTHORIZED')
print('Condition-level descriptive summaries         : AUTHORIZED')
print('Primary D-vs-A analysis                       : AUTHORIZED')
print('Mandatory D-vs-B/C/E/F analyses               : AUTHORIZED')
print('Frozen secondary-metric D-vs-A analyses       : AUTHORIZED')
print('2,000 paired question bootstrap               : AUTHORIZED')
print('Gene-stratified descriptive reporting         : AUTHORIZED')
print('New endpoints / new arm comparisons           : PROHIBITED')
print('Gene-specific inferential tests               : PROHIBITED')

print('\\nCELL 7C15 FROZEN OUTPUTS')
for label, path in OUTPUTS.items():
    print(f'{label:<46}: {path}')
    print(f'{"SHA-256":<46}: {sha256_file(path)}')

print(f'\\nQC checks                                      : {total_checks}/{total_checks} PASS')

print('\\nNEXT AUTHORIZED CELL')
print('Stage 7C — Cell 7C16                          : prespecified Experiment 2 RAG performance analysis')
print('This is the first cell authorized to calculate arm-performance results.')

print(f'\\nFINAL DECISION                                : {terminal_decision}')
print(separator)

\n==============================================================================================================================================================
EXPERIMENT 2 — STAGE 7C — CELL 7C15
AGGREGATION, PERFORMANCE, AND PAIRED-BOOTSTRAP AUTHORIZATION
Notebook                                      : 22_GES_Aware_Genomic_RAG_Cell_7C15_Aggregation_Performance_and_Bootstrap_Authorization.ipynb
Project root                                  : /content/drive/MyDrive/GES_RAG_Temporal_Study
\nCELL 7C14 REVERIFICATION
Cell 7C14 manifest SHA-256                    : a0691be85e714d259cd863c86a5bb1f8fbfbe8d9a65a221a13d99bd30dfb84fd
Cell 7C14 terminal PASS verified              : YES
Unblinded response rows                       : 1,440
Question-condition units                      : 480
Runs per question-condition                   : 3
Scientific metric values inspected in 7C15   : NO
\nFROZEN ANALYSIS DESIGN
Primary endpoint                              : automated_evidence_fidelity_pass
Pri